In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("customers.csv")

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

### Handle Missing Data Values

In [ ]:
#Data Preprocessing
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="median")
df["Income"]=imputer.fit_transform(df[["Income"]]).ravel()

In [ ]:
df

In [ ]:
df.isnull().sum()

### Feature Engineering

In [ ]:
df["Age"]=2026-df["Year_Birth"]

In [ ]:
df

In [ ]:
df.head(20)

In [ ]:
df.describe()

In [ ]:
df["Dt_Customer"]=pd.to_datetime(df["Dt_Customer"],dayfirst=True)
reference_date=df["Dt_Customer"].max()
df["Customer_Tenure_Days"]=(reference_date-df["Dt_Customer"]).dt.days

In [ ]:
df

In [ ]:
print(df["Dt_Customer"].dtype)

In [ ]:
df["Dt_Customer"].max()

In [ ]:
df.columns

In [ ]:
df["Total_Spending"]=df["MntWines"]+df["MntFruits"]+df["MntMeatProducts"]+df["MntFishProducts"]+df["MntSweetProducts"]+df["MntGoldProds"]

In [ ]:
df

In [ ]:
df["Total_children"]=df["Kidhome"]+df["Teenhome"]

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df["Total_children"].max()

In [ ]:
df["Education"].value_counts()

In [ ]:
df["Education"]=df["Education"].replace({
    "Basic":"Undergraduate",
    "Graduation":"Graduate",
    "Master":"Postgraduate",
    "PhD":"Postgraduate",
    "2n Cycle":"Undergraduate"
})

In [ ]:
df["Education"].value_counts()

In [ ]:
df["Marital_Status"].value_counts()

In [ ]:
df["Living_with"]=df["Marital_Status"].replace({"Married":"Partner","Together":"Partner","Single":"Alone","Divorced":"Alone","Widow":"Alone","Absurd":"Alone","YOLO":"Alone"})

In [ ]:
df["Living_with"]

In [ ]:
df

In [ ]:
df.columns

In [ ]:
cols=["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spending_cols=["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]
cols_to_drop=cols+spending_cols
df_cleaned=df.drop(columns=cols_to_drop)

In [ ]:
cols_to_drop

In [ ]:
df_cleaned

In [ ]:
#Outliers
sns.boxplot(data=df_cleaned,y="Age")

In [ ]:
sns.boxplot(data=df,y="Total_children")

In [ ]:
sns.boxplot(data=df_cleaned,y="Total_Spending")

In [ ]:
sns.boxplot(data=df_cleaned,y="Customer_Tenure_Days")

In [ ]:
sns.boxplot(data=df_cleaned,y="Income")

In [ ]:
cols=["Income","Recency","Response","Age","Total_Spending","Total_children"]

In [ ]:
sns.pairplot(df_cleaned[cols])

In [ ]:
print("Data Size without outlier removal:",len(df_cleaned))


In [ ]:
#Remove outliers 
df_cleaned=df_cleaned[(df_cleaned["Age"]<90)]
df_cleaned=df_cleaned[(df_cleaned["Income"]<600000)]

In [ ]:
print("Data Size with outlier removal:",len(df_cleaned))

In [ ]:
sns.pairplot(df_cleaned[cols])

In [ ]:
num_cols=df_cleaned.select_dtypes(include="number")

In [ ]:
num_cols

In [ ]:
corr_matrix=num_cols.corr()

In [ ]:
corr_matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix,annot=True,fmt=".2f",cmap="coolwarm",annot_kws={"size":7})
plt.tight_layout()

# Feature Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe=OneHotEncoder()
cat_cols=["Education","Living_with"]
enc_cols=ohe.fit_transform(df_cleaned[cat_cols])

In [ ]:
enc_df=pd.DataFrame(enc_cols.toarray(),columns=ohe.get_feature_names_out(cat_cols),index=df_cleaned.index)

In [ ]:
df_encoded=pd.concat([df_cleaned.drop(columns=cat_cols),enc_df],axis=1)

In [ ]:
df_encoded

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
df_scaled=scaler.fit_transform(df_encoded)

In [ ]:
df_scaled.shape

In [ ]:
#2D Visulization
from sklearn.decomposition import PCA
pca=PCA(n_components=2,random_state=42)

In [ ]:
X_pca=pca.fit_transform(df_scaled)

In [ ]:
sns.scatterplot(x=X_pca[:,0],y=X_pca[:,1])

In [ ]:
pca.explained_variance_ratio_

In [ ]:
pca2=PCA(n_components=3,random_state=42)

In [ ]:
X_pca2=pca2.fit_transform(df_scaled)
pca2.explained_variance_ratio_

In [ ]:
#Plot
plt.style.use("dark_background")
fig=plt.figure(figsize=(10,8))
ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca2[:,0],X_pca2[:,1],X_pca2[:,2])
ax.set_xlabel("PCA1--------->")
ax.set_ylabel("PCA2--------->")
ax.set_zlabel("PCA3--------->")
ax.set_title("3d Projection")

In [ ]:
plt.style.available

In [ ]:
fig=plt.figure(figsize=(10,8))
ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca2[:,0],X_pca2[:,1],X_pca2[:,2])

In [ ]:
pca2.explained_variance_ratio_

# Analyze K value

## Elbow Method

In [ ]:
X_pca2

In [ ]:
wcss=[]
from sklearn.cluster import KMeans
from kneed import KneeLocator
for k in range(1,21):
    kmeans=KMeans(n_clusters=k,random_state=42)
    kmeans.fit(X_pca2)
    wcss.append(kmeans.inertia_)

In [ ]:
knee=KneeLocator(range(1,21),wcss,curve="convex",direction="decreasing")
print("Optimal K=",knee.knee)
sns.lineplot(x=range(1,21),y=wcss,marker="o")

## Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score
scores=[]
for k in range(2,21):
    kmeans=KMeans(n_clusters=k,random_state=42)
    labels=kmeans.fit_predict(X_pca2)
    scores.append(silhouette_score(X_pca2,labels))

In [ ]:
scores

In [ ]:
plt.plot(range(2,21),scores,marker="o")

In [ ]:
#Combined Plot
k_range=range(2,21)
fig,ax1=plt.subplots(figsize=(10,8))
ax1.plot(k_range,wcss[:len(k_range)],marker="o",color="blue")
ax1.set_xlabel("K------->")
ax1.set_ylabel("WCSS-------->")
ax2=ax1.twinx()
ax2.plot(k_range,scores[:len(k_range)],marker="x",color="red",linestyle="--")
ax2.set_ylabel("SS-------->")

In [ ]:
import numpy as np
print(type(wcss))
print(np.array(wcss).shape)

In [ ]:
kmeans=KMeans(n_clusters=4,random_state=42)
labels_kmeans=kmeans.fit_predict(X_pca2)

In [ ]:
fig=plt.figure(figsize=(10,8))
ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca2[:,0],X_pca2[:,1],X_pca2[:,2],c=labels_kmeans)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
agg=AgglomerativeClustering(n_clusters=4,linkage='ward')
labels_agg=agg.fit_predict(X_pca2)

In [ ]:
fig=plt.figure(figsize=(10,8))
ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca2[:,0],X_pca2[:,1],X_pca2[:,2],c=labels_agg)

# Characterization of Clusters

In [ ]:
df_encoded["clusters"]=labels_agg

In [ ]:
df_encoded

In [ ]:
pal=["red","blue","yellow","green"]
sns.countplot(x=df_encoded["clusters"],palette=pal,hue=df_encoded["clusters"])

In [ ]:
#Income & Spending Patterns
sns.scatterplot(x=df_encoded["Total_Spending"],y=df_encoded["Income"],hue=df_encoded["clusters"],palette=pal)

In [ ]:
#Cluster Summary
cluster_summary=df_encoded.groupby("clusters").mean()
print(cluster_summary)